In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn 
from sklearn.feature_extraction.text import TfidfVectorizer
import torch.optim as optim
from sklearn.model_selection import train_test_split
import re
import random
import optuna
from torch.cuda.amp import autocast, GradScaler

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

In [ ]:
from datasets import load_dataset

wiki = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True
)

texts = []

for i, doc in enumerate(wiki):
    texts.append(doc["text"])
    if i == 150000:   
        break


In [ ]:
len(texts)

In [ ]:
texts

In [ ]:

def clean_text(text):

    text = text.lower()
    text = re.sub(r'references', ' ', text)
    text = re.sub(r'external links', ' ', text)
    text = re.sub(r'cite', ' ', text)

    text = re.sub(r"http\S+", " ", text)

    text = re.sub(r"\[\d+\]", " ", text)

    text = re.sub(r"[^a-z\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()


In [ ]:
cleaned_texts = []

for text in texts:

    text = clean_text(text)

    words = text.split()

    if 50 <= len(words) <= 1000:
        cleaned_texts.append(text)


In [ ]:
len(cleaned_texts)

In [ ]:
train , temp = train_test_split(cleaned_texts,test_size= 0.2,random_state=42)

val,test = train_test_split(temp , test_size=.5)

In [ ]:
from sklearn.feature_extraction import text

custom_stopwords = [
    "list", "lists", "refers", "known", "used", "following",
    "include", "including", "also", "one", "two", "first",
    "second", "new", "may", "many"
]

stop_words = list(text.ENGLISH_STOP_WORDS.union(custom_stopwords))

In [ ]:
vectorizer = TfidfVectorizer(
  max_features= 6000,
  stop_words= stop_words,
  min_df = 30,
  max_df= .6 ,
  ngram_range= (1,2)
)

In [ ]:
X_train = vectorizer.fit_transform(train).toarray()

In [ ]:
X_val = vectorizer.transform(val).toarray()
X_test = vectorizer.transform(test).toarray()

In [ ]:
class TextDataset(Dataset):
    def __init__(self, data):
        self.data = torch.tensor(data, dtype=torch.float32)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [ ]:
train_dataset = TextDataset(X_train)
val_dataset = TextDataset(X_val)
test_dataset = TextDataset(X_test)

In [ ]:
class TopicNN(nn.Module):

    def __init__(self,input_dim,topics,enc_layers,enc_neurons,dec_layers,dec_neurons,dropout):

        super().__init__()

        # ---------- Encoder ----------
        encoder = []
        in_dim = input_dim

        for i in range(enc_layers):

            encoder.append(nn.Linear(in_dim, enc_neurons))
            encoder.append(nn.BatchNorm1d(enc_neurons))
            encoder.append(nn.ReLU())
            encoder.append(nn.Dropout(dropout))

            in_dim = enc_neurons

        encoder.append(nn.Linear(enc_neurons, topics))

        self.encoder = nn.Sequential(*encoder)

        # ---------- Decoder ----------
        decoder = []
        in_dim = topics

        for i in range(dec_layers):

            decoder.append(nn.Linear(in_dim, dec_neurons))
            decoder.append(nn.ReLU())
            decoder.append(nn.Dropout(dropout))

            in_dim = dec_neurons

        decoder.append(nn.Linear(in_dim, input_dim))

        self.decoder = nn.Sequential(*decoder)

    def forward(self, x):
        logits = self.encoder(x)
        z      = torch.softmax(logits/ 0.3, dim=-1)
        recon  = self.decoder(z)
        return recon, z


In [ ]:
def topic_diversity_loss(z):
    z = nn.functional.normalize(z, dim=1)
    sim = torch.matmul(z, z.T)
    return sim.mean()

In [ ]:
criterion = nn.KLDivLoss(reduction='batchmean')

In [ ]:
def objective(trial):

    
    enc_layers  = trial.suggest_int('enc_layers', 1, 3)
    enc_neurons = trial.suggest_int('enc_neurons', 128, 512, step=128)
    dec_layers  = trial.suggest_int('dec_layers', 1, 3)
    dec_neurons = trial.suggest_int('dec_neurons', 128, 512, step=128)
    topics      = trial.suggest_int('topics', 20, 75)

    
    dropout      = trial.suggest_float('dropout', 0.1, 0.4)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-4, log=True)

    
    lr         = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [64, 128])

    
    opt_name = trial.suggest_categorical(
        'optimizer', ['Adam', 'AdamW', 'RMSprop']
    )

    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, pin_memory=True)


    model = TopicNN(
        input_dim=X_train.shape[1],
        topics=topics,
        enc_layers=enc_layers,
        enc_neurons=enc_neurons,
        dec_layers=dec_layers,
        dec_neurons=dec_neurons,
        dropout=dropout
    ).to(device)

    
    opt_map = {
        'Adam': optim.Adam,
        'AdamW': optim.AdamW,
        'RMSprop': optim.RMSprop
    }

    optimizer = opt_map[opt_name](
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_val = float('inf')

    
    for epoch in range(6):

        # TRAIN
        model.train()
        for batch in train_loader:
            batch = batch.to(device, non_blocking=True)

            
            batch = batch / (batch.sum(dim=1, keepdim=True) + 1e-10)

            recon, z = model(batch)
            log_probs = torch.log_softmax(recon, dim=-1)

            loss = criterion(log_probs, batch) + 0.1 * topic_diversity_loss(z)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # VALIDATION
        model.eval()
        total_loss, total_samples = 0, 0

        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device, non_blocking=True)
                batch = batch / (batch.sum(dim=1, keepdim=True) + 1e-10)

                recon, _ = model(batch)
                log_probs = torch.log_softmax(recon, dim=-1)

                loss = criterion(log_probs, batch)

                total_loss += loss.item() * batch.size(0)
                total_samples += batch.size(0)

        val_loss = total_loss / total_samples

        # Optuna pruning
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        best_val = min(best_val, val_loss)

    return best_val

In [ ]:
pruner = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3)




In [ ]:
study  = optuna.create_study(direction='minimize', pruner=pruner)


In [ ]:
study.optimize(objective, n_trials=30)

In [ ]:
print('Best Val Loss:', study.best_value)
print('Best Params:',  study.best_params)

In [ ]:
best = study.best_params

In [ ]:
print(best)

In [ ]:
EPOCHS = 20
PATIENCE = 4
DIV_WEIGHT = 0.3

train_loader = DataLoader(train_dataset, batch_size=best['batch_size'], shuffle=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=best['batch_size'], pin_memory=True)

model = TopicNN(
    X_train.shape[1],
    best['topics'],
    best['enc_layers'],
    best['enc_neurons'],
    best['dec_layers'],
    best['dec_neurons'],
    best['dropout']
).to(device)


opt_map = {
    'Adam': optim.Adam,
    'AdamW': optim.AdamW,
    'RMSprop': optim.RMSprop
}

optimizer = opt_map[best.get('optimizer', 'AdamW')](
    model.parameters(),
    lr=best['lr'],
    weight_decay=best['weight_decay']
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2)
scaler = GradScaler()

best_val, pat = float('inf'), 0

for epoch in range(EPOCHS):

    model.train()
    for batch in train_loader:
        batch = batch.to(device, non_blocking=True)

       
        batch = batch / (batch.sum(dim=1, keepdim=True) + 1e-10)

        
        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=device.type == 'cuda'):
            recon, z = model(batch)
            log_probs = torch.log_softmax(recon, dim=-1)

            loss = criterion(log_probs, batch) + DIV_WEIGHT * topic_diversity_loss(z)

        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

    
    model.eval()
    total_loss, total_samples = 0, 0

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device, non_blocking=True)
            batch = batch / (batch.sum(dim=1, keepdim=True) + 1e-10)

            recon, _ = model(batch)
            log_probs = torch.log_softmax(recon, dim=-1)

            loss = criterion(log_probs, batch)

            total_loss += loss.item() * batch.size(0)
            total_samples += batch.size(0)

    val_loss = total_loss / total_samples
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1} | Val Loss: {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        pat = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        pat += 1
        if pat >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break


model.load_state_dict(torch.load("best_model.pt"))

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=best['batch_size'])

model.eval()
total_loss, total_samples = 0, 0

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        batch = batch / (batch.sum(dim=1, keepdim=True) + 1e-10)

        recon, _ = model(batch)
        log_probs = torch.log_softmax(recon, dim=-1)

        loss = criterion(log_probs, batch)

        total_loss += loss.item() * batch.size(0)
        total_samples += batch.size(0)

print("Test Loss:", total_loss / total_samples)

In [ ]:
def reconstruction_accuracy(model, loader, k=10):
    model.eval()
    total_score = 0
    total_samples = 0

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            batch = batch / (batch.sum(dim=1, keepdim=True) + 1e-10)

            recon, _ = model(batch)

            true_topk = torch.topk(batch, k=k, dim=1).indices
            pred_topk = torch.topk(recon, k=k, dim=1).indices

            for t, p in zip(true_topk, pred_topk):
                overlap = len(set(t.cpu().numpy()) & set(p.cpu().numpy()))
                total_score += overlap / k

            total_samples += batch.size(0)

    return total_score / total_samples

In [ ]:
train_loader_eval = DataLoader(train_dataset, batch_size=best['batch_size'])
val_loader_eval   = DataLoader(val_dataset, batch_size=best['batch_size'])
test_loader_eval  = DataLoader(test_dataset, batch_size=best['batch_size'])

train_acc = reconstruction_accuracy(model, train_loader_eval)
val_acc   = reconstruction_accuracy(model, val_loader_eval)
test_acc  = reconstruction_accuracy(model, test_loader_eval)

print(f"\nReconstruction Accuracy:")
print(f"Train Acc: {train_acc:.4f}")
print(f"Val Acc:   {val_acc:.4f}")
print(f"Test Acc:  {test_acc:.4f}")

In [ ]:
def get_top_words(model, vectorizer, n_top=10):

    vocab = vectorizer.get_feature_names_out()
    n_topics = best['topics']

    eye = torch.eye(n_topics, device=device)

    model.eval()
    with torch.no_grad():
        word_logits = model.decoder(eye)
        word_probs = torch.softmax(word_logits, dim=-1).cpu().numpy()

    
    word_probs = word_probs / (word_probs.sum(axis=1, keepdims=True) + 1e-10)

    print(f'\nTop {n_top} words per topic:\n')

    for i, probs in enumerate(word_probs):
        top_idx = probs.argsort()[-n_top:][::-1]
        top_words = [vocab[j] for j in top_idx]

        print(f'Topic {i+1:02d}: {", ".join(top_words)}')

In [ ]:
def infer_topics(texts_list, model, vectorizer, top_k=5):

    cleaned = [clean_text(t) for t in texts_list]

    X = vectorizer.transform(cleaned).toarray().astype(np.float32)

    
    norms = X.sum(axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)
    X = X / norms

    tensor = torch.tensor(X, device=device)

    model.eval()
    with torch.no_grad():
        _, z = model(tensor)

       
        z = torch.softmax(z, dim=1)

    z = z.cpu().numpy()

    for i, doc in enumerate(texts_list):
        top_topics = z[i].argsort()[-top_k:][::-1]

        print(f'\nDoc {i+1}: {doc[:80]}...')
        for t in top_topics:
            print(f'  Topic {t+1:02d}: {z[i][t]:.3f}')

    return z

In [ ]:

get_top_words(model, vectorizer)
topic_matrix = infer_topics(test[:3], model, vectorizer)

In [ ]:
infer_topics([
    "machine learning models are used for prediction and data analysis",
    "the indian government passed a new law in parliament",
    "football teams compete in international tournaments and leagues",
    "bollywood movies and actors are famous in india"
], model, vectorizer)

In [ ]:
infer_topics([
   "Sunrisers Hyderabad (SRH) put up a united effort, including Nitish Kumar’s all-round show and Heinrich Klaasen’s solid half-century, to record a convincing 65-run victory over Kolkata Knight Riders (KKR) and open its account in IPL 2026 at the Eden Gardens here on Thursday (April 2, 2026)."
], model, vectorizer)